# 03 · FONDPATRIMDISVAR: financial contract profiling

**Objective:** verify the FONDPATRIMDISVAR contract on the real Silver data: grain, coverage, units/signs and the accounting identities. This notebook does not recompute official CNMV metrics or clean rows.

**Scope.** `fund_period_financials` built from the six FONDPATRIMDISVAR files in Bronze (202306–202512), cross-checked against `fund_nav_daily` and `fund_metadata` at the same closures. `as_of_period` is pinned to 202512.

Structural checks live in `validate_silver.py`; this notebook focuses on what those checks cannot decide: whether units and identities make sense and which differences must be documented rather than blocked.


In [1]:
import duckdb

ROOT = "../.."  # repo root, relative to this notebook's folder (notebooks/exploration)
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW fin AS
    SELECT * REPLACE (CAST(FechaDatos AS VARCHAR) AS FechaDatos) FROM read_parquet('{ROOT}/data/silver/fund_period_financials/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW nav AS
    SELECT * REPLACE (CAST(FechaDatos AS VARCHAR) AS FechaDatos) FROM read_parquet('{ROOT}/data/silver/fund_nav_daily/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW meta AS
    SELECT * REPLACE (CAST(FechaDatos AS VARCHAR) AS FechaDatos) FROM read_parquet('{ROOT}/data/silver/fund_metadata/**/*.parquet', hive_partitioning=true)""")

print("Reproducibility header: slice of Silver measured in this run")
display(
    duckdb.sql("""
    SELECT count(*) AS rows,
           count(DISTINCT FechaDatos) AS closures,
           min(FechaDatos) AS first_period,
           max(FechaDatos) AS last_period,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento)) AS compartments
    FROM fin
""").df()
)
print(
    "columns in the Silver contract:",
    duckdb.sql("SELECT count(*) FROM pragma_table_info('fin')").fetchone()[0],
)

Reproducibility header: slice of Silver measured in this run


,rows,closures,first_period,last_period,compartments
0,10237,6,202306,202512,1970


columns in the Silver contract: 40


## 1. Scale and publication cadence
**Question:** how many rows and compartments arrive per closure, and does the observed cadence match the quarterly description in the explanatory PDF?

In [2]:
print("rows, compartments and period-end AUM per closure")
display(
    duckdb.sql("""
    SELECT FechaDatos AS period,
           count(*) AS rows,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento)) AS compartments,
           round(sum(TotalPatrimonio) / 1e9, 1) AS aum_bn
    FROM fin
    GROUP BY 1 ORDER BY 1
""").df()
)
print(
    "published months:",
    duckdb.sql(
        "SELECT list_sort(list(DISTINCT substr(FechaDatos, 5, 2))) FROM fin"
    ).fetchone()[0],
)
print(
    "non-FI rows:",
    duckdb.sql("SELECT count(*) FROM fin WHERE Tipo <> 'FI'").fetchone()[0],
)

rows, compartments and period-end AUM per closure


,period,rows,compartments,aum_bn
0,202306,1712,1712,337.7
1,202312,1715,1715,353.3
2,202406,1722,1722,379.8
3,202412,1714,1714,405.9
4,202506,1706,1706,428.2
5,202512,1668,1668,458.7


published months: ['06', '12']
non-FI rows: 0


- **Note:** the table has six closures, 10.237 rows and 1.970 distinct compartments; every row is FI. AUM grows from 337,7 to 458,7 bn € across the six files, so the series is not a tiny sample.
  - The PDF says **quarterly (03/06/09/12)**, but the downloaded CNMV archives only contain **06/12**; there are no 03/09 files to parse in 2023–2025.
  - Each closure is one row per compartment, not per class: the financial statements are reported at compartment level.
- **Hypothesis:** the semiannual coverage is a property of the published individual-information package, not a parser filter; the PDF describes the intended quarterly file while the package ships it at the half-year closings.
- **Gate:** the parser and gate accept 03/06/09/12 defensively, but dbt treats the observed cadence as 06/12 and never fabricates missing quarters (`fee_period_status` is `missing` when there is no row).

## 2. Coverage and referential integrity
**Question:** do the financial compartments join cleanly to the monthly NAV and metadata tables, and how many compartments are present in one Silver table but not the others?

In [3]:
print("compartment coverage against metadata and daily NAV")
display(
    duckdb.sql("""
    WITH f AS (
        SELECT DISTINCT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento FROM fin
    ), m AS (
        SELECT DISTINCT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento FROM meta
    ), n AS (
        SELECT DISTINCT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento FROM nav
    ), periods AS (SELECT DISTINCT FechaDatos FROM f)
    SELECT p.FechaDatos AS period,
           (SELECT count(*) FROM f WHERE f.FechaDatos = p.FechaDatos) AS fin_comps,
           (SELECT count(*) FROM m WHERE m.FechaDatos = p.FechaDatos) AS meta_comps,
           (SELECT count(*) FROM n WHERE n.FechaDatos = p.FechaDatos) AS nav_comps,
           (SELECT count(*) FROM (SELECT * FROM f WHERE f.FechaDatos = p.FechaDatos
              EXCEPT SELECT * FROM m WHERE m.FechaDatos = p.FechaDatos)) AS fin_not_meta,
           (SELECT count(*) FROM (SELECT * FROM m WHERE m.FechaDatos = p.FechaDatos
              EXCEPT SELECT * FROM f WHERE f.FechaDatos = p.FechaDatos)) AS meta_not_fin,
           (SELECT count(*) FROM (SELECT * FROM f WHERE f.FechaDatos = p.FechaDatos
              EXCEPT SELECT * FROM n WHERE n.FechaDatos = p.FechaDatos)) AS fin_not_nav,
           (SELECT count(*) FROM (SELECT * FROM n WHERE n.FechaDatos = p.FechaDatos
              EXCEPT SELECT * FROM f WHERE f.FechaDatos = p.FechaDatos)) AS nav_not_fin
    FROM periods p ORDER BY 1
""").df()
)
print(
    "period-end AUM reconciliation: fund_period_financials vs last daily NAV per compartment"
)
display(
    duckdb.sql("""
    WITH last_nav AS (
        SELECT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento,
               sum(Patrimonio) AS nav_aum
        FROM nav n
        WHERE value_date = (SELECT max(value_date) FROM nav n2 WHERE n2.FechaDatos = n.FechaDatos)
        GROUP BY 1, 2, 3, 4
    ), recon AS (
        SELECT f.FechaDatos, f.TotalPatrimonio, l.nav_aum,
               abs(f.TotalPatrimonio - l.nav_aum) AS diff
        FROM fin f JOIN last_nav l USING (FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento)
    )
    SELECT FechaDatos AS period, count(*) AS compartments,
           count(*) FILTER (WHERE diff <= 2000) AS within_2000,
           count(*) FILTER (WHERE diff > greatest(2000, 0.01 * abs(TotalPatrimonio))) AS material,
           round(quantile_cont(diff::DOUBLE, 0.50), 2) AS p50_diff,
           round(quantile_cont(diff::DOUBLE, 0.95), 2) AS p95_diff,
           round(max(diff), 2) AS max_diff
    FROM recon GROUP BY 1 ORDER BY 1
""").df()
)

compartment coverage against metadata and daily NAV


,period,fin_comps,meta_comps,nav_comps,fin_not_meta,meta_not_fin,fin_not_nav,nav_not_fin
0,202306,1712,1721,1712,0,9,0,0
1,202312,1715,1719,1715,0,4,0,0
2,202406,1722,1730,1723,0,8,0,1
3,202412,1714,1723,1714,0,9,0,0
4,202506,1706,1708,1706,0,2,0,0
5,202512,1668,1674,1668,0,6,0,0


period-end AUM reconciliation: fund_period_financials vs last daily NAV per compartment


,period,compartments,within_2000,material,p50_diff,p95_diff,max_diff
0,202306,1712,1708,2,0.33,258.88,22413758.67
1,202312,1715,1713,1,0.34,202.90,26335579.43
2,202406,1722,1719,1,0.34,204.54,18834468.93
3,202412,1714,1706,2,0.34,191.45,10156049.33
4,202506,1706,1698,1,0.34,212.65,46613151.51
5,202512,1668,1665,1,0.34,214.71,54320915.17


- **Note:** the financial layer never creates an orphan.
  - Every financial compartment exists in metadata and in daily NAV; `fin_not_meta` and `fin_not_nav` are zero in all six closures.
  - The reverse direction has small holes: 2–9 metadata compartments per closure and one 202406 NAV compartment carry no financial row. These are missing reports, not parse failures.
  - AUM reconciles almost exactly: median difference 0,33 €, 99,8 % of compartments within 2.000 €, and only 1–2 material exceptions per closure.
- **Hypothesis:** the holes are funds or compartments without a complete financial report at that closure; the 0,33 € median is decimal rounding between class-level NAV and compartment-level financial statements.
- **Gate:** dbt joins financials with `LEFT JOIN` on the compartment key and counts missing rows. The `recon_error` metric uses the material threshold, not exact equality; material exceptions are annotated, not silently dropped.

## 3. Units, signs and nullability
**Question:** which fields are sparse, which signs does the source use, and do the monetary and percentage fields arrive in the units the PDF describes?

In [4]:
print("null counts in the fields that are nullable in the XSD")
display(
    duckdb.sql("""
    SELECT 'IndiceRotacionCarteraAnterior' AS field,
           count(*) FILTER (WHERE IndiceRotacionCarteraAnterior IS NULL) AS nulls FROM fin
    UNION ALL SELECT 'Suscripciones_Reembolsos_Netos',
           count(*) FILTER (WHERE Suscripciones_Reembolsos_Netos IS NULL) FROM fin
    UNION ALL SELECT 'BeneficiosBrutosDistribuidos',
           count(*) FILTER (WHERE BeneficiosBrutosDistribuidos IS NULL) FROM fin
    UNION ALL SELECT 'ResultadosRentaFija',
           count(*) FILTER (WHERE ResultadosRentaFija IS NULL) FROM fin
    ORDER BY nulls DESC
""").df()
)
print(
    "nulls, negative counts and percentiles of the main variation fields (percentage points)"
)
display(
    duckdb.sql("""
    SELECT 'IndiceRotacionCarteraActual' AS field,
           count(*) - count(IndiceRotacionCarteraActual) AS nulls,
           count(*) FILTER (WHERE IndiceRotacionCarteraActual < 0) AS negatives,
           round(quantile_cont(IndiceRotacionCarteraActual::DOUBLE, 0.05), 2) AS p05,
           round(quantile_cont(IndiceRotacionCarteraActual::DOUBLE, 0.50), 2) AS p50,
           round(quantile_cont(IndiceRotacionCarteraActual::DOUBLE, 0.95), 2) AS p95,
           round(min(IndiceRotacionCarteraActual::DOUBLE), 2) AS min_value,
           round(max(IndiceRotacionCarteraActual::DOUBLE), 2) AS max_value
    FROM fin
    UNION ALL SELECT 'Suscripciones_Reembolsos_Netos',
           count(*) - count(Suscripciones_Reembolsos_Netos),
           count(*) FILTER (WHERE Suscripciones_Reembolsos_Netos < 0),
           round(quantile_cont(Suscripciones_Reembolsos_Netos::DOUBLE, 0.05), 2),
           round(quantile_cont(Suscripciones_Reembolsos_Netos::DOUBLE, 0.50), 2),
           round(quantile_cont(Suscripciones_Reembolsos_Netos::DOUBLE, 0.95), 2),
           round(min(Suscripciones_Reembolsos_Netos::DOUBLE), 2),
           round(max(Suscripciones_Reembolsos_Netos::DOUBLE), 2)
    FROM fin
    UNION ALL SELECT 'RendimientosNetos',
           count(*) - count(RendimientosNetos),
           count(*) FILTER (WHERE RendimientosNetos < 0),
           round(quantile_cont(RendimientosNetos::DOUBLE, 0.05), 2),
           round(quantile_cont(RendimientosNetos::DOUBLE, 0.50), 2),
           round(quantile_cont(RendimientosNetos::DOUBLE, 0.95), 2),
           round(min(RendimientosNetos::DOUBLE), 2),
           round(max(RendimientosNetos::DOUBLE), 2)
    FROM fin
    UNION ALL SELECT 'GastosRepercutidos',
           count(*) - count(GastosRepercutidos),
           count(*) FILTER (WHERE GastosRepercutidos < 0),
           round(quantile_cont(GastosRepercutidos::DOUBLE, 0.05), 2),
           round(quantile_cont(GastosRepercutidos::DOUBLE, 0.50), 2),
           round(quantile_cont(GastosRepercutidos::DOUBLE, 0.95), 2),
           round(min(GastosRepercutidos::DOUBLE), 2),
           round(max(GastosRepercutidos::DOUBLE), 2)
    FROM fin
    UNION ALL SELECT 'ComisionGestion',
           count(*) - count(ComisionGestion),
           count(*) FILTER (WHERE ComisionGestion < 0),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.05), 2),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.50), 2),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.95), 2),
           round(min(ComisionGestion::DOUBLE), 2),
           round(max(ComisionGestion::DOUBLE), 2)
    FROM fin
    UNION ALL SELECT 'BeneficiosBrutosDistribuidos',
           count(*) - count(BeneficiosBrutosDistribuidos),
           count(*) FILTER (WHERE BeneficiosBrutosDistribuidos < 0),
           round(quantile_cont(BeneficiosBrutosDistribuidos::DOUBLE, 0.05), 2),
           round(quantile_cont(BeneficiosBrutosDistribuidos::DOUBLE, 0.50), 2),
           round(quantile_cont(BeneficiosBrutosDistribuidos::DOUBLE, 0.95), 2),
           round(min(BeneficiosBrutosDistribuidos::DOUBLE), 2),
           round(max(BeneficiosBrutosDistribuidos::DOUBLE), 2)
    FROM fin
    ORDER BY field
""").df()
)
print(
    "sign convention: expenses negative, distributions mostly zero, income mostly positive"
)
display(
    duckdb.sql("""
    SELECT 'GastosRepercutidos' AS field,
           count(*) FILTER (WHERE GastosRepercutidos > 0) AS positive,
           count(*) FILTER (WHERE GastosRepercutidos < 0) AS negative FROM fin
    UNION ALL SELECT 'ComisionGestion', count(*) FILTER (WHERE ComisionGestion > 0), count(*) FILTER (WHERE ComisionGestion < 0) FROM fin
    UNION ALL SELECT 'ComisionDepositario', count(*) FILTER (WHERE ComisionDepositario > 0), count(*) FILTER (WHERE ComisionDepositario < 0) FROM fin
    UNION ALL SELECT 'GastosServiciosExteriores', count(*) FILTER (WHERE GastosServiciosExteriores > 0), count(*) FILTER (WHERE GastosServiciosExteriores < 0) FROM fin
    UNION ALL SELECT 'OtrosGastosGestion', count(*) FILTER (WHERE OtrosGastosGestion > 0), count(*) FILTER (WHERE OtrosGastosGestion < 0) FROM fin
    UNION ALL SELECT 'OtrosGastosRepercutidos', count(*) FILTER (WHERE OtrosGastosRepercutidos > 0), count(*) FILTER (WHERE OtrosGastosRepercutidos < 0) FROM fin
    UNION ALL SELECT 'Ingresos', count(*) FILTER (WHERE Ingresos > 0), count(*) FILTER (WHERE Ingresos < 0) FROM fin
    UNION ALL SELECT 'BeneficiosBrutosDistribuidos', count(*) FILTER (WHERE BeneficiosBrutosDistribuidos > 0), count(*) FILTER (WHERE BeneficiosBrutosDistribuidos < 0) FROM fin
    ORDER BY field
""").df()
)
print("monetary scale: period-end patrimony percentiles (EUR)")
display(
    duckdb.sql("""
    SELECT round(quantile_cont(TotalPatrimonio::DOUBLE, 0.50) / 1e6, 1) AS p50_m,
           round(quantile_cont(TotalPatrimonio::DOUBLE, 0.95) / 1e6, 1) AS p95_m,
           round(max(TotalPatrimonio::DOUBLE) / 1e9, 1) AS max_bn,
           count(*) FILTER (WHERE TotalPatrimonio <= 0) AS nonpositive
    FROM fin
""").df()
)

null counts in the fields that are nullable in the XSD


,field,nulls
0,IndiceRotacionCarteraAnterior,439
1,Suscripciones_Reembolsos_Netos,1
2,BeneficiosBrutosDistribuidos,1
3,ResultadosRentaFija,1


nulls, negative counts and percentiles of the main variation fields (percentage points)


,field,nulls,negatives,p05,p50,p95,min_value,max_value
0,BeneficiosBrutosDistribuidos,1,233,0.00,0.00,0.00,-13.54,0.91
1,ComisionGestion,1,10142,-1.78,-0.57,-0.10,-16.12,0.00
2,GastosRepercutidos,1,10185,-2.24,-0.73,-0.17,-313.97,0.00
3,IndiceRotacionCarteraActual,1,1,0.00,0.19,1.48,-0.19,117.53
4,RendimientosNetos,1,943,-1.62,3.53,17.61,-280.27,83.69
5,Suscripciones_Reembolsos_Netos,1,5687,-37.00,-1.15,127.11,-1254.97,1997.86


sign convention: expenses negative, distributions mostly zero, income mostly positive


,field,positive,negative
0,BeneficiosBrutosDistribuidos,1,233
1,ComisionDepositario,0,10067
2,ComisionGestion,0,10142
3,GastosRepercutidos,0,10185
4,GastosServiciosExteriores,0,7251
5,Ingresos,3106,3
6,OtrosGastosGestion,0,4325
7,OtrosGastosRepercutidos,1,8273


monetary scale: period-end patrimony percentiles (EUR)


,p50_m,p95_m,max_bn,nonpositive
0,41.7,1052.9,20.6,1


- **Note:** the contract is complete and the source units hold.
  - Only four fields have nulls: `IndiceRotacionCarteraAnterior` (439, concentrated in the 2023 closures), and one null each in `Suscripciones_Reembolsos_Netos`, `BeneficiosBrutosDistribuidos` and `ResultadosRentaFija`.
  - Expenses are **negative** (`GastosRepercutidos` 10.185 negative, `ComisionGestion` 10.142, `ComisionDepositario` 10.067); `Ingresos` is positive in 3.106 rows and negative in 3.
  - `BeneficiosBrutosDistribuidos` is zero in 10.002 rows and negative when a distribution reduces patrimony; it is not a positive income.
  - Variation fields are percentages, not fractions: `RendimientosNetos` median 3,53, p95 17,61 and extremes -280,27 / 83,69. `Suscripciones_Reembolsos_Netos` ranges from -1.254,97 to 1.997,86. `TotalPatrimonio` is monetary: median 41,7 M€, p95 1,05 bn €, maximum 20,6 bn €, with one non-positive value.
- **Hypothesis:** the negative expenses and distributions follow the CNMV sign convention; the null in the previous-turnover field is more common in the first closure of each year because the field is optional. The extreme percentage values come from tiny compartments where a small amount is a huge percentage.
- **Gate:** Silver keeps signs and percentages exactly as published. dbt uses `abs()` for expense yields, tests the identities below, and sends extreme returns/flows to the anomaly layer instead of clipping them in Silver.

## 4. Accounting identities and reconciliation
**Question:** do the identities defined in the explanatory document hold, and which residuals are rounding noise versus a material source inconsistency?

In [5]:
identities = [
    (
        "IF = CI + CE + IC + ID",
        "abs(DPInversionesFinancieras - (CarteraInterior + CarteraExterior + InteresesCartera + InversionesDudosas))",
        "monetary",
        2000,
    ),
    (
        "TP = IF + L + R",
        "abs(TotalPatrimonio - (DPInversionesFinancieras + Liquidez + Resto))",
        "monetary",
        2000,
    ),
    (
        "GR = CGES + CDEP + GSE + OGGC + OGR",
        "abs(GastosRepercutidos - (ComisionGestion + ComisionDepositario + GastosServiciosExteriores + OtrosGastosGestion + OtrosGastosRepercutidos))",
        "pp",
        0.05,
    ),
    (
        "ING = CDFAV + CRETRO + OING",
        "abs(Ingresos - (ComisionesDescuento + ComisionesRetrocedidas + OtrosIngresos))",
        "pp",
        0.05,
    ),
    (
        "RN = RG + GR + ING",
        "abs(RendimientosNetos - (RendimientosGestion + GastosRepercutidos + Ingresos))",
        "pp",
        0.05,
    ),
    (
        "RG = INT + DIV + RRF + RRV + RDEP + RDER + RIIC + ORES + OREN",
        "abs(RendimientosGestion - (Intereses + Dividendos + ResultadosRentaFija + ResultadosRentaVariable + ResultadosDepositos + ResultadosDerivados + ResultadosIIC + OtrosResultados + OtrosRendimientos))",
        "pp",
        0.05,
    ),
    (
        "TP = PatrimonioFinPeriodoActual",
        "abs(TotalPatrimonio - PatrimonioFinPeriodoActual)",
        "monetary",
        2000,
    ),
]
identity_sql = " UNION ALL ".join(
    f"""SELECT '{label}' AS identity, '{unit}' AS unit, {tol} AS tolerance,
        count(*) FILTER (WHERE residual > {tol}) AS over_tolerance,
        round(max(residual), 2) AS max_residual
        FROM (SELECT {expr} AS residual FROM fin)"""
    for label, expr, unit, tol in identities
)
print("identity residuals and tolerance (monetary: EUR; pp: percentage points)")
display(duckdb.sql(identity_sql).df())
print("largest material monetary residuals")
display(
    duckdb.sql("""
    SELECT * FROM (
        SELECT 'TP = IF + L + R' AS identity, FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento,
               round(TotalPatrimonio - (DPInversionesFinancieras + Liquidez + Resto), 2) AS residual
        FROM fin
        WHERE abs(TotalPatrimonio - (DPInversionesFinancieras + Liquidez + Resto)) > 2000
        UNION ALL
        SELECT 'TP = PatrimonioFinPeriodoActual', FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento,
               round(TotalPatrimonio - PatrimonioFinPeriodoActual, 2)
        FROM fin
        WHERE abs(TotalPatrimonio - PatrimonioFinPeriodoActual) > 2000
    ) ORDER BY abs(residual) DESC
    LIMIT 8
""").df()
)

identity residuals and tolerance (monetary: EUR; pp: percentage points)


,identity,unit,tolerance,over_tolerance,max_residual
0,IF = CI + CE + IC + ID,monetary,2000.00,0,3.200000e+01
1,TP = IF + L + R,monetary,2000.00,1,1.683369e+09
2,GR = CGES + CDEP + GSE + OGGC + OGR,pp,0.05,1,1.200000e-01
3,ING = CDFAV + CRETRO + OING,pp,0.05,0,1.000000e-02
4,RN = RG + GR + ING,pp,0.05,2,1.200000e-01
5,RG = INT + DIV + RRF + RRV + RDEP + RDER + RII...,pp,0.05,1,9.000000e-02
6,TP = PatrimonioFinPeriodoActual,monetary,2000.00,47,1.011481e+09


largest material monetary residuals


,identity,FechaDatos,Tipo,NumeroRegistro,NumeroCompartimento,residual
0,TP = IF + L + R,202412,FI,1189,0,1.683369e+09
1,TP = PatrimonioFinPeriodoActual,202512,FI,737,1,-1.011481e+09
2,TP = PatrimonioFinPeriodoActual,202306,FI,1404,0,1.197197e+08
3,TP = PatrimonioFinPeriodoActual,202406,FI,4807,0,5.053968e+07
4,TP = PatrimonioFinPeriodoActual,202306,FI,5314,0,1.360169e+07
5,TP = PatrimonioFinPeriodoActual,202512,FI,737,5,-6.331018e+06
6,TP = PatrimonioFinPeriodoActual,202512,FI,737,6,-5.605145e+06
7,TP = PatrimonioFinPeriodoActual,202512,FI,737,4,-4.561056e+06


- **Note:** the income-statement identities are sound; the monetary balances mostly round to thousands.
  - `GR`, `ING`, `RN` and `RG` exceed 0,05 pp in only 1, 0, 2 and 1 rows respectively (maximum 0,12 pp). The statement is internally coherent.
  - `IF = CI + CE + IC + ID` has 360 residuals of 0,01–32,00 €, but none above 2.000 €. `TP = IF + L + R` has 1.016 small residuals and **one material exception** where the distribution fields are zero and `TotalPatrimonio` is 1,68 bn €.
  - `TP = PatrimonioFinPeriodoActual` is **not a stable identity**: 3.860 rows differ by more than 0,02 € and 47 by more than 2.000 €; the largest residual is 1,01 bn €.
- **Hypothesis:** small residuals are rounding of the reported components to thousands of euros. The material outliers are source-reporting inconsistencies (a distribution snapshot with zero components, and a stale/duplicated period-end patrimony), not a cast or join error.
- **Gate:** dbt tests `GR`, `ING`, `RN` and `RG` at 0,05 pp. Monetary balances use an absolute tolerance of 2.000 € and a relative materiality flag; `TP = PatrimonioFinPeriodoActual` is documented and flagged as a reconciliation anomaly, never used as a blocking equality test.

## 5. Comparability across closures
**Question:** can the six closures be compared directly, or do the cumulative percentages reset and change meaning between June and December?

In [6]:
print("cumulative variation fields by closure (percentage points)")
display(
    duckdb.sql("""
    SELECT FechaDatos AS period, count(*) AS rows,
           round(quantile_cont(RendimientosNetos::DOUBLE, 0.50), 2) AS rn_p50,
           round(quantile_cont(RendimientosNetos::DOUBLE, 0.95), 2) AS rn_p95,
           round(100.0 * count(*) FILTER (WHERE RendimientosNetos < 0) / count(*), 1) AS rn_negative_pct,
           round(quantile_cont(Suscripciones_Reembolsos_Netos::DOUBLE, 0.50), 2) AS flow_p50,
           round(quantile_cont(Suscripciones_Reembolsos_Netos::DOUBLE, 0.95), 2) AS flow_p95,
           round(quantile_cont(GastosRepercutidos::DOUBLE, 0.50), 2) AS gr_p50,
           round(quantile_cont(IndiceRotacionCarteraActual::DOUBLE, 0.50), 2) AS turnover_p50
    FROM fin GROUP BY 1 ORDER BY 1
""").df()
)

cumulative variation fields by closure (percentage points)


,period,rows,rn_p50,rn_p95,rn_negative_pct,flow_p50,flow_p95,gr_p50,turnover_p50
0,202306,1712,2.61,14.88,7.8,-1.04,143.20,-0.53,0.20
1,202312,1715,6.45,20.65,3.7,-2.92,145.06,-1.06,0.16
2,202406,1722,2.52,12.27,9.1,-1.83,91.31,-0.56,0.20
3,202412,1714,4.97,19.28,5.5,-2.21,126.56,-1.10,0.17
4,202506,1706,1.36,11.70,22.3,-0.34,82.99,-0.55,0.21
5,202512,1668,3.83,22.33,6.8,1.39,134.16,-1.12,0.18


- **Note:** the level of every variation field moves with the reporting horizon.
  - December 2024 has a median net return of 4,97 % against 2,52 % in June 2024; December 2025 has 3,83 % against 1,36 % in June 2025. These are **year-to-date** percentages since the previous year-end, not comparable quarter returns.
  - Net flows are negative in five of the six closures (median -1,15 % overall); the exception is December 2025 (median +1,39 %). The share of negative net returns is 3,7–22,3 %, so the cross-section changes materially with the market.
  - Turnover is a half-year figure in every observed 06/12 file: median 0,16–0,21. The PDF's quarter semantics for 03/09 cannot be exercised with the available archive.
- **Hypothesis:** June accumulates roughly six months and December roughly twelve, so the median return naturally grows with the horizon; the 2025 dispersion reflects the market path, not a different calculation.
- **Gate:** dbt preserves the YTD label and compares `Suscripciones_Reembolsos_Netos` only with daily net flows accumulated from the previous year-end at 06/12. It never annualizes these percentages by multiplying by 2 or 4, and it does not compare a single quarter with a YTD value.

## 6. Findings and implications (conclusion)

This table records what the data supports, what must be flagged and what remains a documented limitation.

**Implications**

| Finding | Evidence | Implication |
|---|---|---|
| Grain and keys are sound | 10.237 rows; six closures; one row per compartment; no duplicate or null keys | Unique + not-null dbt tests on `(Tipo, NumeroRegistro, NumeroCompartimento, FechaDatos)` ([§1](#1-scale-and-publication-cadence)) |
| Financials have no orphan compartments | `fin_not_meta = 0`, `fin_not_nav = 0` in all closures | Relationship tests; `LEFT JOIN` metadata and NAV, count missing rows ([§2](#2-coverage-and-referential-integrity)) |
| Some reported compartments have no financial row | 2–9 metadata compartments per closure; one 202406 NAV compartment | `fee_period_status = missing`; no synthetic rows ([§2](#2-coverage-and-referential-integrity)) |
| Nominal quarterly, observed semiannual | Only 06/12 files in the 2023–2025 archive | dbt treats 06/12 as the analysis cadence; parser/gate still accept 03/09 ([§1](#1-scale-and-publication-cadence)) |
| Signs and units are source contracts | Expenses negative; variations in percentage points; patrimony in EUR | Keep signs; `abs()` for yields; divide by 100 only at presentation ([§3](#3-units-signs-and-nullability)) |
| Income-statement identities hold | `GR/ING/RN/RG` exceed 0,05 pp in 1/0/2/1 rows | dbt tests at 0,05 pp; exceptions go to the anomaly layer ([§4](#4-accounting-identities-and-reconciliation)) |
| Monetary balances need tolerance | `IF/TP` residuals are rounding except one 1,68 bn exception | 2.000 € absolute tolerance plus materiality flag; do not block the build on the outlier ([§4](#4-accounting-identities-and-reconciliation)) |
| `TP = PatrimonioFinPeriodoActual` is not enforceable | 3.860 rows above 0,02 €; 47 above 2.000 €; maximum 1,01 bn | Reconciliation anomaly only; never a blocking dbt test ([§4](#4-accounting-identities-and-reconciliation)) |
| Extreme returns and flows exist | Net return -280,27 / 83,69; net flow -1.254,97 / 1.997,86 | Anomaly layer (`mart_fund_anomalies`); no clipping in Silver ([§3](#3-units-signs-and-nullability), [§5](#5-comparability-across-closures)) |
| Four fields are optional | 439 null previous turnover; 1 null in flow, distributions and fixed-income results | dbt counts nulls; no row is discarded ([§3](#3-units-signs-and-nullability)) |
